# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
DATASET_NAME = "merfish"
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_50.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,50,0.574783,0.676753,0.00,0.0,0.00,0.019022,7.927304,6.275710,613.618408
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,50,0.656297,0.672887,0.20,0.9,0.18,0.293658,12.283512,14.924376,1490.194336
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,50,0.301952,0.354013,0.18,1.0,0.18,0.069700,11.340351,12.519340,1868.877075
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,50,0.774857,0.715261,0.26,1.0,0.26,0.000000,23.067350,23.667232,2971.760254
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,50,0.742315,0.662692,0.06,1.0,0.06,0.003236,11.601857,11.063035,1668.539429


In [3]:
# Rename holdout_celltype by replacing the last '_' in with '-'
df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)

In [4]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,50,0.574783,0.676753,0.00,0.0,0.00,0.019022,7.927304,6.275710,613.618408,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,50,0.656297,0.672887,0.20,0.9,0.18,0.293658,12.283512,14.924376,1490.194336,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,50,0.301952,0.354013,0.18,1.0,0.18,0.069700,11.340351,12.519340,1868.877075,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,50,0.774857,0.715261,0.26,1.0,0.26,0.000000,23.067350,23.667232,2971.760254,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,50,0.742315,0.662692,0.06,1.0,0.06,0.003236,11.601857,11.063035,1668.539429,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,C57BL6J-2.041,spatialprop,astrocyte,50,0.689508,0.743634,0.10,1.0,0.10,0.030505,7.641941,10.864837,3015.643800,Fiber-tracts
234,C57BL6J-2.041,spatialprop,oligodendrocyte,50,0.405234,0.650548,0.12,1.0,0.12,0.156538,7.753944,10.421536,1406.491800,Isocortex
235,C57BL6J-2.041,spatialprop,oligodendrocyte,50,0.267803,0.625423,0.06,1.0,0.06,0.044337,15.737027,17.804389,7421.856400,Fiber-tracts
236,C57BL6J-2.041,spatialprop,endothelial cell,50,0.813205,0.887102,0.28,1.0,0.28,0.999113,7.615954,8.076030,2622.968000,Isocortex


In [6]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "direction_match_k": +1,
    "edistance_local": -1,
    #"rmse": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "spatialprop": "SpatialProp",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "SpatialProp",
    "MintFlow",
    "cellina (ablated)",
    "cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina (graph)',
       'CPA', 'scGen', 'MintFlow', 'SpatialProp'], dtype=object)

In [7]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [8]:
agg

spearman             pearson            \
                                    mean       std      mean       std   
perturbation model_name                                                  
Fiber-tracts mean shift         0.351411  0.271832  0.360952  0.238833   
             CPA                0.584916  0.206990  0.798208  0.149858   
             scGen              0.496118  0.229757  0.715875  0.201174   
             SpatialProp        0.471744  0.204791  0.639798  0.199540   
             MintFlow           0.546033  0.228744  0.779940  0.179947   
             cellina (ablated)  0.584583  0.215306  0.797372  0.141151   
             cellina (graph)    0.635315  0.142233  0.807087  0.153321   
             cellina            0.670275  0.221507  0.831716  0.151117   
Isocortex    mean shift         0.594199  0.188672  0.575376  0.191781   
             CPA                0.781948  0.170337  0.837973  0.172484   
             scGen              0.719862  0.182515  0.820247  0.142527   
             SpatialProp        0.704227  0.123463  0.745628  0.093146   
             MintFlow           0.771883  0.184344  0.836692  0.164884   
             cellina (ablated)  0.789042  0.162555  0.841151  0.163441   
             cellina (graph)    0.845406  0.105102  0.888711  0.141875   
             cellina            0.816266  0.178495  0.861217  0.170908   

                               precision           direction_match_k  \
                                    mean       std              mean   
perturbation model_name                                                
Fiber-tracts mean shift         0.070667  0.068813          0.068000   
             CPA                0.318667  0.127720          0.309333   
             scGen              0.150667  0.088759          0.149333   
             SpatialProp        0.086667  0.074322          0.086667   
             MintFlow           0.220000  0.156570          0.205333   
             cellina (ablated)  0.292000  0.142588          0.280000   
             cellina (graph)    0.397143  0.170584          0.385714   
             cellina            0.500000  0.231887          0.492000   
Isocortex    mean shift         0.256000  0.077164          0.250667   
             CPA                0.534667  0.136584          0.534667   
             scGen              0.268000  0.161917          0.268000   
             SpatialProp        0.089333  0.078873          0.088000   
             MintFlow           0.369333  0.156728          0.366667   
             cellina (ablated)  0.502667  0.138743          0.501333   
             cellina (graph)    0.627143  0.113844          0.627143   
             cellina            0.614667  0.164311          0.614667   

                                         edistance_local            
                                     std            mean       std  
perturbation model_name                                             
Fiber-tracts mean shift         0.069200       10.321689  3.513345  
             CPA                0.125554        2.519827  2.521286  
             scGen              0.090354        2.993651  2.519029  
             SpatialProp        0.074322       10.414518  5.814760  
             MintFlow           0.162035       15.813728  5.674291  
             cellina (ablated)  0.141825        2.666084  2.787318  
             cellina (graph)    0.173547        2.570265  2.431353  
             cellina            0.236740        2.275430  2.133649  
Isocortex    mean shift         0.081369       18.318042  4.656952  
             CPA                0.136584        4.818402  3.751915  
             scGen              0.161917        5.853639  4.366436  
             SpatialProp        0.078486       12.965062  2.176068  
             MintFlow           0.161186       17.701288  2.162100  
             cellina (ablated)  0.139277        5.123815  4.330667  
             cellina (graph)    0.113844        4.674285  3.158653  
             c

In [9]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s

In [10]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.351 $\pm$ 0.272   
             CPA                         0.585 $\pm$ 0.207   
             scGen                       0.496 $\pm$ 0.230   
             SpatialProp                 0.472 $\pm$ 0.205   
             MintFlow                    0.546 $\pm$ 0.229   
             cellina (ablated)           0.585 $\pm$ 0.215   
             cellina (graph)             0.635 $\pm$ 0.142   
             cellina            \textbf{0.670 $\pm$ 0.222}   
Isocortex    mean shift                  0.594 $\pm$ 0.189   
             CPA                         0.782 $\pm$ 0.170   
             scGen                       0.720 $\pm$ 0.183   
             SpatialProp                 0.704 $\pm$ 0.123   
             MintFlow                    0.772 $\pm$ 0.184   
             cellina (ablated)           0.789 $\pm$ 0.163   
             cellina (graph)    \textbf{0.845 $\pm$ 0.105}   
             cellina                     0.816 $\pm$ 0.178   

                                        Pearson $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.361 $\pm$ 0.239   
             CPA                         0.798 $\pm$ 0.150   
             scGen                       0.716 $\pm$ 0.201   
             SpatialProp                 0.640 $\pm$ 0.200   
             MintFlow                    0.780 $\pm$ 0.180   
             cellina (ablated)           0.797 $\pm$ 0.141   
             cellina (graph)             0.807 $\pm$ 0.153   
             cellina            \textbf{0.832 $\pm$ 0.151}   
Isocortex    mean shift                  0.575 $\pm$ 0.192   
             CPA                         0.838 $\pm$ 0.172   
             scGen                       0.820 $\pm$ 0.143   
             SpatialProp                 0.746 $\pm$ 0.093   
             MintFlow                    0.837 $\pm$ 0.165   
             cellina (ablated)           0.841 $\pm$ 0.163   
             cellina (graph)    \textbf{0.889 $\pm$ 0.142}   
             cellina                     0.861 $\pm$ 0.171   

                                      Precision $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.071 $\pm$ 0.069   
             CPA                         0.319 $\pm$ 0.128   
             scGen                       0.151 $\pm$ 0.089   
             SpatialProp                 0.087 $\pm$ 0.074   
             MintFlow                    0.220 $\pm$ 0.157   
             cellina (ablated)           0.292 $\pm$ 0.143   
             cellina (graph)             0.397 $\pm$ 0.171   
             cellina            \textbf{0.500 $\pm$ 0.232}   
Isocortex    mean shift                  0.256 $\pm$ 0.077   
             CPA                         0.535 $\pm$ 0.137   
             scGen                       0.268 $\pm$ 0.162   
             SpatialProp                 0.089 $\pm$ 0.079   
             MintFlow                    0.369 $\pm$ 0.157   
             cellina (ablated)           0.503 $\pm$ 0.139   
             cellina (graph)    \textbf{0.627 $\pm$ 0.114}   
             cellina                     0.615 $\pm$ 0.164   

                               Direction Match (K) $\uparrow$  \
perturbation model_name                                         
Fiber-tracts mean shift                     0.068 $\pm$ 0.069   
             CPA                            0.309 $\pm$ 0.126   
             scGen                          0.149 $\pm$ 0.090   
             SpatialProp                    0.087 $\pm$ 0.074   
             MintFlow                       0.205 $\pm$ 0.162   
             cellina (ablated)              0.280 $\pm$ 0.142   
             cellina (graph)                0.386 $\pm$ 0.174   
             cellina               \textbf{0.492 $\pm$ 0.237}   
Isocortex    mean shift               

In [11]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [12]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:{FILE_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:{FILE_NAME}",
    )

print(latex)

(OUT_DIR / f"{FILE_NAME}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish_DEG_50}
\begin{tabular}{llccccc}
\toprule
Holdout \\ perturbation & Method & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & Direction Match (K) $\uparrow$ & E-dist (local) $\downarrow$ \\
\toprule

\multirow[c]{8}{*}{Fiber-tracts} & mean shift & 0.351 $\pm$ 0.272 & 0.361 $\pm$ 0.239 & 0.071 $\pm$ 0.069 & 0.068 $\pm$ 0.069 & 10.322 $\pm$ 3.513 \\
 & CPA & 0.585 $\pm$ 0.207 & 0.798 $\pm$ 0.150 & 0.319 $\pm$ 0.128 & 0.309 $\pm$ 0.126 & 2.520 $\pm$ 2.521 \\
 & scGen & 0.496 $\pm$ 0.230 & 0.716 $\pm$ 0.201 & 0.151 $\pm$ 0.089 & 0.149 $\pm$ 0.090 & 2.994 $\pm$ 2.519 \\
 & SpatialProp & 0.472 $\pm$ 0.205 & 0.640 $\pm$ 0.200 & 0.087 $\pm$ 0.074 & 0.087 $\pm$ 0.074 & 10.415 $\pm$ 5.815 \\
 & MintFlow & 0.546 $\pm$ 0.229 &

2545

In [13]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                                | Spearman ↑    | Pearson ↑     | Precision ↑   | Direction Match (K) ↑   | E-dist (local) ↓   |
|:--------------------------------------|:--------------|:--------------|:--------------|:------------------------|:-------------------|
| ('Fiber-tracts', 'mean shift')        | 0.351 ± 0.272 | 0.361 ± 0.239 | 0.071 ± 0.069 | 0.068 ± 0.069           | 10.322 ± 3.513     |
| ('Fiber-tracts', 'CPA')               | 0.585 ± 0.207 | 0.798 ± 0.150 | 0.319 ± 0.128 | 0.309 ± 0.126           | 2.520 ± 2.521      |
| ('Fiber-tracts', 'scGen')             | 0.496 ± 0.230 | 0.716 ± 0.201 | 0.151 ± 0.089 | 0.149 ± 0.090           | 2.994 ± 2.519      |
| ('Fiber-tracts', 'SpatialProp')       | 0.472 ± 0.205 | 0.640 ± 0.200 | 0.087 ± 0.074 | 0.087 ± 0.074           | 10.415 ± 5.815     |
| ('Fiber-tracts', 'MintFlow')          | 0.546 ± 0.229 | 0.780 ± 0.180 | 0.220 ± 0.157 | 0.205 ± 0.162           | 15.814 ± 5.674     |
| ('Fiber-tracts', 'cellina (ablated)') |